In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install roboflow

from roboflow import Roboflow
# Store your Roboflow API key in Colab Secrets as ROBOFLOW_API_KEY
from google.colab import userdata
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("my-workspace-00").project("my-last-ds-s7ut5")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to my-last-DS-1 in yolov8:: 100%|██████████| 58730/58730 [00:15<00:00, 3787.88it/s]


In [3]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 23.0/107.7 GB disk)


In [ ]:
from ultralytics import YOLO
import os

data_path = os.path.join(dataset.location, "/content/my-last-DS-1/data.yaml")

model = YOLO('yolov8n.pt')

model.train (
    data=data_path,
    epochs=40,
    imgsz=640,
    project='/content/drive/MyDrive/YOLO_Final_Project',
    batch=16,
    device=0,
    patience=10 ,
    name='weapon_model',
    resume=False
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/my-last-DS-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=weapon_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspec

In [6]:
from ultralytics import YOLO
import os
import cv2
from google.colab.patches import cv2_imshow

# 1. Load the model from your Google Drive
# Ensure this path matches your actual file location
model_path = '/content/drive/MyDrive/YOLO_Final_Project/weapon_model/weights/last.pt'
model = YOLO(model_path)

# 2. Run prediction on the test images
# conf=0.25 helps detect smaller/distant objects that the model is less certain about
results = model.predict(
    source='/content/my-last-DS-1/test/images',
    conf=0.25,
    save=True
)

print("--- Searching for detections in test images ---")

displayed_count = 0
for r in results:
    # Only show images where at least one object was detected
    if len(r.boxes) > 0:
        # Plot the boxes on the image
        combined_img = r.plot()

        print(f"Detected in: {os.path.basename(r.path)}")
        cv2_imshow(combined_img)

        displayed_count += 1

    # Limit display to 10 images to keep the notebook clean
    if displayed_count >= 10:
        break

if displayed_count == 0:
    print("No detections found. Try lowering the 'conf' parameter to 0.15.")

In [7]:
from ultralytics import YOLO

# 1. Load your model
model_path = '/content/drive/MyDrive/YOLO_Final_Project/weapon_model/weights/last.pt'
model = YOLO(model_path)

# 2. Run Validation
# This will evaluate the model on the validation set defined in your data.yaml
metrics = model.val(data='/content/my-last-DS-1/data.yaml')

# 3. Print the most important results
print("\n" + "="*30)
print("FINAL EVALUATION RESULTS:")
print(f"Mean Average Precision (mAP@50): {metrics.box.map50:.4f}")
print(f"Mean Average Precision (mAP@50-95): {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print("="*30)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1287.6±375.1 MB/s, size: 64.2 KB)
val: Scanning /content/my-last-DS-1/valid/labels.cache... 2066 images, 147 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2066/2066 481.4Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 3, len(boxes) = 2091. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.1s/it 8:51
                   all       2066       2091      0.908      0.846      0.901      0.603
Speed: 2.4ms preprocess, 249.0ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

file_name = '/content/Manual_Evaluation_Sheet3.xlsx'
df = pd.read_excel(file_name)


df['True_Count (Fill Me)'] = pd.to_numeric(df['True_Count (Fill Me)'], errors='coerce')
df = df.dropna(subset=['True_Count (Fill Me)'])

y_true = df['True_Count (Fill Me)']
y_pred = df['Predicted_Count']

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)

print("="*40)
print("       FINAL REGRESSION EVALUATION       ")
print("="*40)
print(f"Total Images Audited: {len(df)}")
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print("="*40)

df['Error'] = df['Predicted_Count'] - df['True_Count (Fill Me)']
print("\nImages with Errors:")
print(df[df['Error'] != 0][['Image_Name', 'True_Count (Fill Me)', 'Predicted_Count', 'Error']])

       FINAL REGRESSION EVALUATION       
Total Images Audited: 637
Mean Squared Error (MSE):  0.5181
Mean Absolute Error (MAE): 0.3391

Images with Errors:
                                            Image_Name  True_Count (Fill Me)  \
2     1295_png.rf.cf7a429b0222574f637fc9c0282e7f76.jpg                     3   
5    2-246-_jpg.rf.a009362adcd153bcccc3d8adace89f8c...                     1   
6    -12_14_2022-9_19_42-PM_png.rf.dcc589ae5a94c3f6...                     1   
8      -4-_jpg.rf.85af1e0144b26884f903803cefb4ece1.jpg                     0   
9      135_jpg.rf.696b2378ecdb285a88c376193c574350.jpg                     1   
..                                                 ...                   ...   
621  knife_734_jpg.rf.3c5eb87af3a02f12c838a4740f7ee...                     2   
628  istockphoto-465139526-612x612_jpg.rf.3a5a817f0...                     4   
630  images88_jpg.rf.408f92a06b67f8b5b31e520a11a251...                     1   
631  scott-szarapka-dlzKioOq7FA-unsplash_jp

The model was evaluated on an external dataset consisting of 637 images to test its robustness in real-world scenarios. The results show a Mean Absolute Error (MAE) of 0.3391, indicating that the model's predictions are highly accurate, with an average deviation of less than one-third of an object per image. The Mean Squared Error (MSE) of 0.5181 further confirms the model's stability, showing no significant outliers or large miscounting errors across the entire test set. These metrics demonstrate that the model is highly reliable for automated weapon counting and surveillance tasks.